![alt text](icon.png) 

# Pivotal demo

Demo of the Pivotal language and Jupyter Lab extension. Pivotal source code and docs avaialble on [GitHub](https://github.com/nealbob/pivotal-py). Data is from the Kaggle [European Soccer Database](https://www.kaggle.com/datasets/hugomathien/soccer). 



In [1]:
import pivotal
%pivotal_set canvas=a4

Pivotal settings: {'output_type': 'viewer', 'output_code': False, 'canvas': 'a4', 'margins': 25.4, 'chart_width': 'full'}


In [2]:
big4 = ["England Premier League", "Spain LIGA BBVA", "Germany 1. Bundesliga", "Italy Serie A"]

In [3]:
%%pivotal 
load Match "database.sqlite"
load League "database.sqlite"

In [4]:
%%pivotal
df league_names from League
    filter name in :big4
    rename id as league_id, name as League
    select league_id, League

df Match
    inner merge league_naxmes on league_id
    assign season_clean = left(season,4) + "-" + right(season,2)

In [5]:
%%pivotal
df goal_summary from Match
    assign total_goals = home_team_goal + away_team_goal
    pivot
        agg mean total_goals
        rows season_clean
        cols League

In [6]:
%%pivotal
plot line goal_chart
    x season_clean "Season"
    y :big4 "Mean goals"

In [7]:
%%pivotal
table goal_table
    title "Mean goals scored per game by league and season"
    format number 2
    stub season_clean "Season"
    summary mean as "Average"
    font size 12
    font "Calibri"

In [8]:
%%pivotal
load Team "database.sqlite"
    select team_api_id, team_long_name        

In [9]:
%%pivotal
df win_summary from Match
    assign winner_id = home_team_api_id 
        where home_team_goal > away_team_goal
    assign winner_id = away_team_api_id
        where home_team_goal < away_team_goal
    left merge Team
         left_on winner_id
         right_on team_api_id
    assign wins = 1
    group by team_long_name, League
        agg sum wins
    select team_long_name as Team, League, wins as Wins 
    sort Wins asc
    filter Wins > 50

In [10]:
%%pivotal
plot barh win_plot
    x Team 
    y Wins ""
    by League
    legend False

In [11]:
%%pivotal
save "football_demo"

Package 'football_demo' saved to /home/nealh/test_pivotal/football_demo (8 dataframe(s), 2 chart(s), 1 table(s))


In [6]:
Match.columns[0:11]


Index(['id', 'country_id', 'league_id', 'season', 'stage', 'date',
       'match_api_id', 'home_team_api_id', 'away_team_api_id',
       'home_team_goal', 'away_team_goal'],
      dtype='str')